In [ ]:
# Two-stage BCI IV-2a LOSO: enable GPU and Internet, then run.
import os, shutil, subprocess, sys
from pathlib import Path

BRANCH = 'feature/hada-masked-reconstruction-pretrain'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/kaggle/working/tcformer-ssl-pretrain')
SUBJECT_IDS = [1]  # Add more folds only if the Kaggle session budget permits.
PRETRAIN_EPOCHS = 100
MASK_RATIO = 0.4
MASK_SPAN_MS = 100
FREQUENCY_LOSS_WEIGHT = 0.0
SOURCE_ONLY = False
CHECKPOINT_DIR = Path('/kaggle/working/pretrained_encoders/bcic2a')
MNE_DATA = Path('/kaggle/working/mne_data/bciciv2a')
RESULT_ARCHIVE = Path('/kaggle/working/hada_ssl_bcic2a_results')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MNE_DATA.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None, env=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run(['nvidia-smi'])
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED':'1','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','MPLBACKEND':'Agg','MNE_DATA':str(MNE_DATA),'MNE_DATASETS_BNCI_PATH':str(MNE_DATA)})

pretrain_command = [PYTHON, '-u', 'pretrain_tcformer.py', '--dataset', 'bcic2a', '--epochs', PRETRAIN_EPOCHS, '--mask_ratio', MASK_RATIO, '--mask_span_ms', MASK_SPAN_MS, '--frequency_loss_weight', FREQUENCY_LOSS_WEIGHT, '--output_dir', CHECKPOINT_DIR, '--skip_existing', '--subject_ids', *SUBJECT_IDS]
if SOURCE_ONLY: pretrain_command.append('--source_only')
run(pretrain_command, cwd=REPO_PATH, env=environment)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'bcic2a', '--loso', '--gpu_id', '0', '--pretrained_encoder_dir', CHECKPOINT_DIR, '--subject_ids', *SUBJECT_IDS], cwd=REPO_PATH, env=environment)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Two-stage training complete. Kaggle output:', archive, flush=True)
